# Page classifier — BERT training on a PC

The local twin of `colab.ipynb`: same steps, no Drive. Everything stays in this folder:

* input `data/training_data.jsonl` (copy it from the annotation tool's image folder)
* model `output/page-family/` (best epoch so far), resume state `output/page-family-resume/`

Set up once with `setup_pc.ps1 -Notebook` (Windows) or `./setup_pc.sh --notebook` (Mac / Linux), then open this notebook with the `.venv` Python as the kernel (VS Code: *Select Kernel* → `.venv`). Page text is never printed — only counts and metrics.

In [ ]:
import subprocess, sys
from pathlib import Path

HERE = Path.cwd()                           # run the notebook from training/bert-training
DATA = HERE / 'data' / 'training_data.jsonl'
OUTPUT = HERE / 'output' / 'page-family'
RESUME = HERE / 'output' / 'page-family-resume'
SCRIPT = HERE / 'train.py'
assert SCRIPT.is_file(), 'open the notebook from training/bert-training'
print('data present:', DATA.is_file(), '|', DATA)

def train(*args):
    """Run train.py with this folder's paths; its output streams below."""
    cmd = [sys.executable, str(SCRIPT), '--data', str(DATA), '--output', str(OUTPUT),
           '--resume-dir', str(RESUME), *args]
    subprocess.run(cmd, check=True)

Which device will train: an NVIDIA GPU (CUDA) when PyTorch sees one, otherwise the CPU.

In [ ]:
import torch
print('torch', torch.__version__)
print('CUDA GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none')

Compatibility check: two training steps and a minutes-per-epoch estimate. Saves nothing.

In [ ]:
train('--check')

A new run: 4 epochs (3–5 is a good range; the model folder keeps the best epoch, so more never hurts). Use `train('--restart')` instead to throw away an existing run and start over.

In [ ]:
train('--epochs', '4')

One more epoch of the same run (same classes, same validation pages). Repeat while the numbers improve.

In [ ]:
train('--resume', '--epochs', '1')

In [ ]:
import pandas as pd
pd.read_csv(OUTPUT / 'training_log.csv')

In [ ]:
pd.read_csv(OUTPUT / 'category_wise_metrics.csv').sort_values('f1')

When a run is good, copy `output/page-family/` to `core-pipeline/models/page-family/` — the pipeline loads it from there.

## Inference check

Try the saved model (the best epoch) on any text: the top 5 model types with probabilities, and the page type and codability each one means. This is BERT alone — the pipeline also runs the keyword model and the decision ladder, so its final answer can differ, especially for the model types BERT skipped.

In [ ]:
from predict import Classifier

clf = Classifier(OUTPUT)
print(len(clf.labels), 'model types trained;', len(clf.untrained), 'left to the keyword model')

In [ ]:
# Paste a page's text and press Classify.
import ipywidgets as widgets
from IPython.display import display

box = widgets.Textarea(placeholder='Paste page text here', layout=widgets.Layout(width='100%', height='220px'))
button = widgets.Button(description='Classify', button_style='primary')
result = widgets.Output()

def on_click(_):
    with result:
        result.clear_output()
        clf.show(box.value)

button.on_click(on_click)
display(box, button, result)

In [ ]:
# No widgets? Paste the text between the triple quotes and run this cell.
TEXT = """
paste page text here
"""
clf.show(TEXT)